本地训练 → 加权平均 → 全局更新
•	🔁 每一轮训练包含三步循环：
1. 下发：服务器将当前全局模型（如一个轻量版BERT或CNN）发送给部分在线手机（例如：100台中随机选10台）；
2. 本地训练：每台手机用自己的数据（如个人输入习惯、相册图片）进行数轮训练（5–10次SGD），生成本地更新模型；
3. 上传与聚合：手机只上传模型参数的变化量（ΔW），服务器用加权平均法（按各设备数据量比例）融合所有更新，得到新全局模型。

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

In [2]:
# 1.生成数据
X, y = make_classification(n_samples=3000, n_features=10, random_state=42)
clients_X = np.array_split(X, 3)        # 3*client
clients_y = np.array_split(y, 3)

In [3]:
# 2.初始化全局模型参数
global_model = LogisticRegression(max_iter=1, warm_start=True)      # 只做1次
global_model.fit(clients_X[0], clients_y[0])  # 用一个客户端初始化
global_coef = global_model.coef_.copy()             # （1，10）
global_intercept = global_model.intercept_.copy()   # （1，）

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
# 3.FedAvg 参数设定
n_rounds = 10       # 通信轮次
local_epochs = 5    # 每个客户端本地训练轮数

In [5]:
# 4.FedAvg训练
for r in range(n_rounds):
    client_coefs = []
    client_intercepts = []
    client_sizes = []

    # 模拟客户端本地训练
    for i in range(3):
        # 每个客户端从全局模型出发，本地训练
        local_model = LogisticRegression(max_iter=local_epochs, warm_start=True)
        local_model.coef_ = global_coef.copy()
        local_model.intercept_ = global_intercept.copy()
        # 在本地数据上训练（只训练 local_epochs 轮，模拟本地多步 SGD）
        local_model.fit(clients_X[i], clients_y[i])


        client_coefs.append(local_model.coef_)
        client_intercepts.append(local_model.intercept_)
        client_sizes.append(len(clients_y[i]))

    # 服务器加权平均
    total = sum(client_sizes)
    global_coef = sum(c * (n / total) for c, n in zip(client_coefs, client_sizes))
    global_intercept = sum(c * (n / total) for c, n in zip(client_intercepts, client_sizes))

    # 评估全局模型
    # 把聚合后的参数赋给全局模型，然后在全部数据上计算准确率
    global_model.coef_ = global_coef
    global_model.intercept_ = global_intercept
    acc = global_model.score(X, y)
    print(f"第 {r+1} 轮，全局模型准确率: {acc:.4f}")

第 1 轮，全局模型准确率: 0.9733
第 2 轮，全局模型准确率: 0.9743
第 3 轮，全局模型准确率: 0.9743
第 4 轮，全局模型准确率: 0.9743
第 5 轮，全局模型准确率: 0.9743
第 6 轮，全局模型准确率: 0.9743
第 7 轮，全局模型准确率: 0.9743
第 8 轮，全局模型准确率: 0.9743
第 9 轮，全局模型准确率: 0.9743
第 10 轮，全局模型准确率: 0.9743


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to